# Dự báo Xổ Số 5/35 sử dụng Phân Tích Markov và Thống Kê

## Phương pháp:
1. **Phân tích tần suất**: Đếm số lần xuất hiện từng số
2. **Chuỗi Markov**: Phân tích chuyển đổi từ kỳ này sang kỳ tiếp theo
3. **Xác suất có điều kiện**: Tính xác suất dựa trên lịch sử
4. **Entropy và thông tin**: Đo độ không chắc chắn
5. **Dự báo**: Gợi ý các số có xác suất cao nhất
6. **Kiểm dị trùng**: Loại bỏ các số đã xuất hiện gần đây

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Cài đặt hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Các thư viện đã được import thành công")

In [ ]:
# Đọc dữ liệu
df = pd.read_csv('aipl/lotto535.csv')

print("📊 Thông tin dữ liệu:")
print(f"Tổng số kỳ quay: {len(df)}")
print(f"Từ ngày: {df['Ngày'].iloc[0]} đến {df['Ngày'].iloc[-1]}")
print(f"\nCột dữ liệu: {df.columns.tolist()}")
print(f"\nDữ liệu mẫu (5 dòng cuối):")
print(df.tail())

In [ ]:
# ============================================
# PHẦN 1: PHÂN TÍCH TẦN SUẤT VÀ THỐNG KÊ CƠ BẢN
# ============================================

class LotteryAnalyzer:
    def __init__(self, df):
        self.df = df
        self.s1_range = (1, 16)
        self.s2_range = (4, 23)
        self.s3_range = (8, 28)
        self.s4_range = (13, 33)
        self.s5_range = (21, 35)
        self.s6_range = (1, 12)
        
    def calculate_frequency(self):
        """Tính tần suất xuất hiện của mỗi số"""
        freq = {}
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            freq[col] = Counter(self.df[col])
        return freq
    
    def get_frequency_stats(self):
        """Tính toán thống kê tần suất"""
        freq = self.calculate_frequency()
        stats = {}
        
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            col_freq = freq[col]
            values = list(col_freq.values())
            
            stats[col] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values)
            }
        
        return freq, stats

analyzer = LotteryAnalyzer(df)
freq, stats = analyzer.get_frequency_stats()

print("\n📈 THỐNG KÊ TẦN SUẤT:")
print("="*60)

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    s = stats[col]
    print(f"\n{col.upper()}:")
    print(f"  Trung bình: {s['mean']:.2f} lần")
    print(f"  Độ lệch chuẩn: {s['std']:.2f}")
    print(f"  Xuất hiện ít nhất: {s['min']} lần")
    print(f"  Xuất hiện nhiều nhất: {s['max']} lần")

In [ ]:
# ============================================
# PHẦN 2: PHÂN TÍCH CHUỖI MARKOV
# ============================================

class MarkovChain:
    def __init__(self, df, column_name, window_size=5):
        self.df = df
        self.column = column_name
        self.window_size = window_size
        self.transition_matrix = self._build_transition_matrix()
        
    def _build_transition_matrix(self):
        """Xây dựng ma trận chuyển đổi Markov"""
        transitions = defaultdict(lambda: Counter())
        
        values = self.df[self.column].values
        for i in range(len(values) - 1):
            current = values[i]
            next_val = values[i + 1]
            transitions[current][next_val] += 1
        
        return transitions
    
    def get_next_probabilities(self, current_value):
        """Tính xác suất của giá trị tiếp theo dựa trên giá trị hiện tại"""
        if current_value not in self.transition_matrix:
            return {}
        
        transitions = self.transition_matrix[current_value]
        total = sum(transitions.values())
        
        probabilities = {}
        for next_val, count in transitions.items():
            probabilities[next_val] = count / total
        
        return probabilities
    
    def get_top_candidates(self, current_value, top_k=5):
        """Lấy top K ứng viên có xác suất cao nhất"""
        probs = self.get_next_probabilities(current_value)
        if not probs:
            return []
        
        sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
        return sorted_probs[:top_k]

# Tạo chuỗi Markov cho mỗi vị trí
markov_chains = {}
for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    markov_chains[col] = MarkovChain(df, col)

# Lấy giá trị cuối cùng
last_row = df.iloc[-1]
print("\n🔄 PHÂN TÍCH CHUỖI MARKOV:")
print("="*60)
print(f"\nKỳ quay cuối cùng: Kỳ {int(last_row['Kỳ'])} ({last_row['Ngày']})")
print(f"Kết quả: {int(last_row['s1'])} - {int(last_row['s2'])} - {int(last_row['s3'])} - {int(last_row['s4'])} - {int(last_row['s5'])} - {int(last_row['s6'])}")

print("\n📊 XÁC SUẤT CỦA KỲ TIẾP THEO (Dựa trên giá trị cuối):")

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    current_val = int(last_row[col])
    candidates = markov_chains[col].get_top_candidates(current_val, top_k=5)
    
    print(f"\n{col.upper()} (hiện tại: {current_val}):")
    if candidates:
        for val, prob in candidates:
            print(f"  {val}: {prob*100:.1f}%")
    else:
        print(f"  Không có dữ liệu lịch sử cho giá trị {current_val}")

In [ ]:
# ============================================
# PHẦN 3: PHÂN TÍCH ENTROPY VÀ THÔNG TIN
# ============================================

def calculate_entropy(probabilities):
    """Tính entropy Shannon"""
    entropy = 0
    for prob in probabilities.values():
        if prob > 0:
            entropy -= prob * np.log2(prob)
    return entropy

def calculate_information_content(probabilities):
    """Tính lượng thông tin - mức độ bất ngờ"""
    info = {}
    for val, prob in probabilities.items():
        if prob > 0:
            info[val] = -np.log2(prob)
    return info

print("\n📊 PHÂN TÍCH ENTROPY:")
print("="*60)

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    freq_dict = freq[col]
    total = sum(freq_dict.values())
    probabilities = {k: v/total for k, v in freq_dict.items()}
    
    entropy = calculate_entropy(probabilities)
    max_entropy = np.log2(len(probabilities))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    
    print(f"\n{col.upper()}:")
    print(f"  Entropy: {entropy:.3f} (Chuẩn hóa: {normalized_entropy:.3f})")
    print(f"  Giải thích: Entropy cao = phân bố đều, Entropy thấp = phân bố tập trung")

In [ ]:
# ============================================
# PHẦN 4: PHÁT HIỆN SỐ "NÓNG" VÀ "LẠNH"
# ============================================

def analyze_hot_cold_numbers(df, window_size=20):
    """Phân tích các số nóng (xuất hiện nhiều gần đây) và lạnh (xuất hiện ít)"""
    results = {}
    
    recent_df = df.tail(window_size)
    older_df = df[:-window_size] if len(df) > window_size else pd.DataFrame()
    
    for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
        # Tần suất gần đây
        recent_freq = Counter(recent_df[col])
        recent_total = len(recent_df)
        
        # Tần suất cũ
        if len(older_df) > 0:
            older_freq = Counter(older_df[col])
            older_total = len(older_df)
        else:
            older_freq = Counter()
            older_total = 1
        
        hot_cold = {}
        for num in range(1, 40):
            recent_count = recent_freq.get(num, 0)
            older_count = older_freq.get(num, 0)
            
            recent_rate = recent_count / recent_total
            older_rate = older_count / older_total if older_total > 0 else 0
            
            hot_cold[num] = {
                'recent': recent_count,
                'older': older_count,
                'recent_rate': recent_rate,
                'older_rate': older_rate,
                'trend': recent_rate - older_rate
            }
        
        results[col] = hot_cold
    
    return results

hot_cold = analyze_hot_cold_numbers(df, window_size=20)

print("\n🔥 PHÂN TÍCH SỐ NÓNG/LẠNH (20 kỳ gần đây):")
print("="*60)

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    print(f"\n{col.upper()}:")
    
    # Sắp xếp theo trend
    sorted_nums = sorted(hot_cold[col].items(), 
                          key=lambda x: x[1]['trend'], 
                          reverse=True)[:5]
    
    print("  🔥 TOP 5 SỐ NÓNG:")
    for num, data in sorted_nums:
        print(f"    {num}: {data['recent']} lần (gần đây), Trend: +{data['trend']:.3f}")

In [ ]:
# ============================================
# PHẦN 5: PHÂN TÍCH KHOẢNG CÁCH (GAP ANALYSIS)
# ============================================

def analyze_gaps(df, column):
    """Phân tích khoảng cách giữa các lần xuất hiện"""
    gaps = defaultdict(list)
    positions = {}
    
    for idx, val in enumerate(df[column].values):
        if val in positions:
            gap = idx - positions[val]
            gaps[val].append(gap)
        positions[val] = idx
    
    # Tính thống kê cho mỗi số
    gap_stats = {}
    for num, gap_list in gaps.items():
        if gap_list:
            gap_stats[num] = {
                'last_gap': gap_list[-1],
                'avg_gap': np.mean(gap_list),
                'max_gap': np.max(gap_list),
                'min_gap': np.min(gap_list)
            }
    
    return gap_stats

print("\n📊 PHÂN TÍCH KHOẢNG CÁCH XUẤT HIỆN (Gap Analysis):")
print("="*60)
print("(Khoảng cách càng lớn = số càng lâu không xuất hiện = có thể sắp quay)\n")

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    gap_stats = analyze_gaps(df, col)
    
    # Lấy top 5 số có khoảng cách lớn nhất
    top_gaps = sorted(gap_stats.items(), 
                      key=lambda x: x[1]['last_gap'], 
                      reverse=True)[:5]
    
    print(f"\n{col.upper()} - TOP 5 SỐ CÓ KHOẢNG CÁCH LỚN (lâu không xuất hiện):")
    for num, stats in top_gaps:
        print(f"  {num}: {stats['last_gap']} kỳ (Trung bình: {stats['avg_gap']:.1f}, Max: {stats['max_gap']})")

In [ ]:
# ============================================
# PHẦN 6: DỰ BÁOS SỬ DỤNG HỘP KẾT HỢP
# ============================================

class PredictionEngine:
    def __init__(self, df, analyzer, markov_chains, hot_cold):
        self.df = df
        self.analyzer = analyzer
        self.markov_chains = markov_chains
        self.hot_cold = hot_cold
        self.freq, self.stats = analyzer.get_frequency_stats()
        
    def calculate_prediction_score(self, col, number):
        """Tính điểm dự báo cho một số"""
        score = 0
        
        # 1. Tần suất (30% trọng lượng)
        total_appearances = sum(self.freq[col].values())
        freq_score = self.freq[col].get(number, 0) / total_appearances
        score += freq_score * 0.30
        
        # 2. Trend nóng/lạnh (25% trọng lượng)
        trend = self.hot_cold[col].get(number, {}).get('trend', 0)
        trend_score = max(0, trend)  # Chỉ lấy trend dương
        score += trend_score * 100 * 0.25
        
        # 3. Khoảng cách (25% trọng lượng)
        gap_stats = analyze_gaps(self.df, col)
        if number in gap_stats:
            avg_gap = gap_stats[number]['avg_gap']
            last_gap = gap_stats[number]['last_gap']
            gap_score = min(1.0, last_gap / (avg_gap + 1))
            score += gap_score * 0.25
        
        # 4. Xác suất Markov (20% trọng lượng)
        if len(self.df) > 0:
            last_val = int(self.df.iloc[-1][col])
            markov_probs = self.markov_chains[col].get_next_probabilities(last_val)
            markov_score = markov_probs.get(number, 0)
            score += markov_score * 0.20
        
        return score
    
    def predict(self, exclude_recent_days=3):
        """Dự báo các số tiếp theo"""
        predictions = {}
        
        # Lấy các số đã xuất hiện gần đây để loại bỏ
        recent_rows = self.df.tail(exclude_recent_days * 2)
        recent_numbers = {}
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            recent_numbers[col] = set(recent_rows[col].values)
        
        # Dự báo cho mỗi vị trí
        ranges = {
            's1': (1, 16),
            's2': (4, 23),
            's3': (8, 28),
            's4': (13, 33),
            's5': (21, 35),
            's6': (1, 12)
        }
        
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            min_num, max_num = ranges[col]
            scores = {}
            
            for number in range(min_num, max_num + 1):
                # Loại bỏ các số xuất hiện gần đây
                if number in recent_numbers[col]:
                    continue
                
                score = self.calculate_prediction_score(col, number)
                scores[number] = score
            
            # Sắp xếp theo điểm
            top_predictions = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
            predictions[col] = top_predictions
        
        return predictions

# Tạo engine dự báo
engine = PredictionEngine(df, analyzer, markov_chains, hot_cold)
predictions = engine.predict(exclude_recent_days=3)

print("\n🎯 DỰ BÁOS KỲ TIẾP THEO:")
print("="*60)
print("(Loại bỏ các số xuất hiện trong 3 kỳ gần đây)\n")

for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
    print(f"\n{col.upper()} - TOP 5 ỨNG VIÊN:")
    for idx, (num, score) in enumerate(predictions[col], 1):
        print(f"  {idx}. Số {num}: Điểm {score:.3f}")

In [ ]:
# ============================================
# PHẦN 7: GỢI Ý TỐI ƯுUA DỰ BÁOS
# ============================================

def get_best_recommendation(predictions):
    """Chọn phối hợp tốt nhất từ dự báos"""
    best_combination = []
    confidence_scores = []
    
    for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
        top_pred = predictions[col][0]  # Lấy ứng viên top 1
        best_combination.append(int(top_pred[0]))
        confidence_scores.append(top_pred[1])
    
    avg_confidence = np.mean(confidence_scores)
    
    return best_combination, confidence_scores, avg_confidence

best_combo, conf_scores, avg_conf = get_best_recommendation(predictions)

print("\n✨ GỢI Ý PHỐI HỢP TỐT NHẤT:")
print("="*60)
print(f"\n🎰 DỰ BÁOS CHÍNH: {' - '.join(map(str, best_combo[:5]))} : {best_combo[5]}")
print(f"\nĐộ tin cậy trung bình: {avg_conf:.3f} (thang điểm 0-1)")
print(f"\nChi tiết:")

for i, (col, score) in enumerate(zip(['s1', 's2', 's3', 's4', 's5', 's6'], conf_scores)):
    print(f"  {col}: Số {best_combo[i]} (Độ tin cậy: {score:.3f})")

In [ ]:
# ============================================
# PHẦN 8: PHƯƠNG ÁN THAY THẾ VÀ PHÂN TẢN RỦI RO
# ============================================

def generate_alternative_combinations(predictions, num_combinations=5):
    """Tạo các phối hợp thay thế để phân tán rủi ro"""
    combinations = []
    
    for combo_idx in range(num_combinations):
        combo = []
        scores = []
        
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            if combo_idx < len(predictions[col]):
                pred = predictions[col][combo_idx]
                combo.append(int(pred[0]))
                scores.append(pred[1])
            else:
                # Nếu không đủ dự báos, lấy dự báos tiếp theo
                pred = predictions[col][-1]
                combo.append(int(pred[0]))
                scores.append(pred[1])
        
        combinations.append({
            'combination': combo,
            'scores': scores,
            'avg_score': np.mean(scores)
        })
    
    return combinations

alternatives = generate_alternative_combinations(predictions, num_combinations=3)

print("\n💡 PHƯƠNG ÁN THAY THẾ (Phân tán rủi ro):")
print("="*60)

for idx, alt in enumerate(alternatives, 1):
    combo_str = f"{' - '.join(map(str, alt['combination'][:5]))} : {alt['combination'][5]}"
    print(f"\nPhương án {idx}: {combo_str}")
    print(f"Độ tin cậy trung bình: {alt['avg_score']:.3f}")

In [ ]:
# ============================================
# PHẦN 9: BIỂU ĐỒ VÀ TRỰC QUAN HÓA
# ============================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Phân tích Tần suất Xuất hiện - Xổ số 5/35', fontsize=16, fontweight='bold')

for idx, col in enumerate(['s1', 's2', 's3', 's4', 's5', 's6']):
    ax = axes[idx // 3, idx % 3]
    
    # Tạo dữ liệu cho biểu đồ
    numbers = sorted(freq[col].keys())
    counts = [freq[col][n] for n in numbers]
    
    # Vẽ bar chart
    bars = ax.bar(numbers, counts, color='steelblue', edgecolor='navy', alpha=0.7)
    
    # Highlight top predictions
    top_pred_nums = [int(p[0]) for p in predictions[col][:3]]
    for bar, num in zip(bars, numbers):
        if num in top_pred_nums:
            bar.set_color('red')
            bar.set_alpha(0.8)
    
    ax.set_xlabel('Số')
    ax.set_ylabel('Tần suất')
    ax.set_title(f'{col.upper()} - Tần suất xuất hiện')
    ax.grid(axis='y', alpha=0.3)
    
    # Thêm đường trung bình
    avg_freq = np.mean(counts)
    ax.axhline(y=avg_freq, color='green', linestyle='--', alpha=0.5, label=f'Trung bình: {avg_freq:.1f}')
    ax.legend()

plt.tight_layout()
plt.savefig('aipl/lottery_frequency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Biểu đồ tần suất đã được lưu: lottery_frequency_analysis.png")

In [ ]:
# ============================================
# PHẦN 10: KIỂM ĐỊNH MÔ HÌNH (Model Validation)
# ============================================

def validate_predictions_backtest(df, lookback_periods=10):
    """Kiểm định độ chính xác của mô hình trên dữ liệu lịch sử"""
    results = {
        'exact_matches': 0,
        'partial_matches': 0,
        'total_tests': 0
    }
    
    for test_idx in range(len(df) - lookback_periods, len(df)):
        # Dùng dữ liệu trước test_idx để dự báos
        test_df = df.iloc[:test_idx]
        
        if len(test_df) < 20:  # Cần đủ dữ liệu để huấn luyện
            continue
        
        # Tạo analyzer và engine với dữ liệu lịch sử
        test_analyzer = LotteryAnalyzer(test_df)
        test_markov = {}
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            test_markov[col] = MarkovChain(test_df, col)
        
        test_hot_cold = analyze_hot_cold_numbers(test_df, window_size=15)
        test_engine = PredictionEngine(test_df, test_analyzer, test_markov, test_hot_cold)
        test_predictions = test_engine.predict()
        
        # Lấy kết quả thực tế
        actual_row = df.iloc[test_idx]
        
        # Kiểm tra độ trùng khớp
        matches = 0
        for col in ['s1', 's2', 's3', 's4', 's5', 's6']:
            pred_nums = [int(p[0]) for p in test_predictions[col][:3]]
            actual_num = int(actual_row[col])
            
            if actual_num in pred_nums:
                matches += 1
        
        if matches == 6:
            results['exact_matches'] += 1
        if matches >= 3:
            results['partial_matches'] += 1
        
        results['total_tests'] += 1
    
    return results

print("\n📈 KIỂM ĐỊNH MỘTÔ HÌNH (Backtest trên 10 kỳ gần đây):")
print("="*60)

validation = validate_predictions_backtest(df, lookback_periods=10)

if validation['total_tests'] > 0:
    exact_rate = (validation['exact_matches'] / validation['total_tests']) * 100
    partial_rate = (validation['partial_matches'] / validation['total_tests']) * 100
    
    print(f"\nTổng số kỳ kiểm định: {validation['total_tests']}")
    print(f"Trùng khớp hoàn toàn (6/6): {validation['exact_matches']} lần ({exact_rate:.1f}%)")
    print(f"Trùng khớp từng phần (3+/6): {validation['partial_matches']} lần ({partial_rate:.1f}%)")
    print(f"\nGhi chú: Đây là kết quả backtest. Xổ số là trò chơi ngẫu nhiên nên không thể dự báos chính xác 100%.")
else:
    print("\nKhông đủ dữ liệu để kiểm định.")

In [ ]:
# ============================================
# PHẦN 11: BÁO CÁO TỔNG HỢP CUỐI CÙNG
# ============================================

print("\n" + "="*70)
print("📊 BÁO CÁO PHÂN TÍCH VÀ DỰ BÁOS XỔ SỐ 5/35 - MARKOV & THỐNG KÊ")
print("="*70)

print(f"\n📅 Ngày phân tích: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"📊 Dữ liệu từ kỳ 0 đến kỳ {int(df.iloc[-1]['Kỳ'])}")
print(f"📈 Tổng số kỳ phân tích: {len(df)}")

print(f"\n{'='*70}")
print("✨ KẾT QUẢ DỰ BÁOS")
print(f"{'='*70}")

print(f"\n🎰 PHỐI HỢP CHÍNH (GỢI Ý HÀNG ĐẦUÂ):")
combo_str = ' - '.join(map(str, best_combo[:5])) + f" : {best_combo[5]}"
print(f"   {combo_str}")
print(f"   Độ tin cậy: {avg_conf:.1%}")

print(f"\n🔄 PHƯƠNG ÁN THAY THẾ:")
for idx, alt in enumerate(alternatives[1:], 1):
    combo_str = ' - '.join(map(str, alt['combination'][:5])) + f" : {alt['combination'][5]}"
    print(f"   Phương án {idx}: {combo_str}")

print(f"\n{'='*70}")
print("⚠️ CẢNHbáo QUAN TRỌNG:")
print(f"{'='*70}")
print("""   • Xổ số là trò chơi ngẫu nhiên - DỰ BÁOS CHỈ MANG TÍNH THAM KHẢO
   • Không có phương pháp nào có thể dự báos chính xác kết quả xổ số
   • Phân tích này dùng để hiểu xu hướng, không đảm bảo trúng thưởng
   • CHƠI XỔ SỐ MỘTcách CÓ TRÁCH NHIỆM - CHỈ VỚI SỐ TIỀN CÓ THỂ MẤT ĐỐ""")

print(f"\n{'='*70}")
print("📚 PHƯƠNG PHÁP PHÂN TÍCH:")
print(f"{'='*70}")
print("""   1. Phân tích tần suất: Tính toán số lần xuất hiện mỗi số
   2. Chuỗi Markov: Phân tích xác suất chuyển đổi giữa các kỳ
   3. Phân tích nóng/lạnh: Xác định xu hướng gần đây
   4. Gap Analysis: Phân tích khoảng cách xuất hiện
   5. Entropy Shannon: Đo độ bất định của phân bố
   6. Hộp kết hợp: Kết hợp nhiều tiêu chí để dự báos""")

print(f"\n{'='*70}\n")

In [ ]:
# Lưu kết quả vào file
export_data = {
    'Phối hợp chính': [' - '.join(map(str, best_combo[:5])) + f" : {best_combo[5]}"],
    'Độ tin cậy': [f"{avg_conf:.3f}"],
    'Phương án 2': [' - '.join(map(str, alternatives[1]['combination'][:5])) + f" : {alternatives[1]['combination'][5]}"],
    'Phương án 3': [' - '.join(map(str, alternatives[2]['combination'][:5])) + f" : {alternatives[2]['combination'][5]}"],
}

result_df = pd.DataFrame(export_data)
result_df.to_csv('aipl/lottery_prediction_result.csv', index=False, encoding='utf-8-sig')

print("✅ Kết quả dự báos đã được lưu vào: lottery_prediction_result.csv")

## Kết Luận

Phân tích này sử dụng:
- **Chuỗi Markov**: Để tìm xác suất chuyển đổi trạng thái
- **Phân tích thống kê**: Để hiểu phân bố tần suất
- **Phân tích xu hướng**: Để xác định các số "nóng" và "lạnh"
- **Kỹ thuật gap analysis**: Để tìm các số lâu không xuất hiện

**Tuy nhiên**, điều quan trọng phải nhớ là:
- Xổ số là **trò chơi ngẫu nhiên hoàn toàn**
- Không có phương pháp nào có thể dự báos chính xác 100%
- Phân tích này chỉ mang tính **tham khảo và học tập**
- **Hãy chơi một cách có trách nhiệm**